In [1]:
"""
Maize Yield Forecasting in Nigeria
Deep Learning Pipeline with Walk-Forward Validation

Author: [Your Name]
License: MIT
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import warnings
import json
import os
from datetime import datetime

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ==========================================================
# Configuration
# ==========================================================
class Config:
    SEED = 42
    SEQ_LENGTH = 5
    PRED_LENGTH = 1
    BATCH_SIZE = 32
    EPOCHS = 150
    LEARNING_RATE = 1e-3
    PATIENCE = 20
    
    FEATURE_COLS = ['LST', 'NDVI', 'Precipitation', 'Temperature',
                    'year_sin', 'year_cos']
    TARGET_COL = 'value'
    STATE_COL = 'admin_1'
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(Config.SEED)
os.makedirs('results/plots', exist_ok=True)
os.makedirs('models', exist_ok=True)


# ==========================================================
# Data Preparation
# ==========================================================
def create_sequences(df, feature_cols, target_col, state_col, 
                     seq_len=5, pred_len=1):
    """Create sliding window sequences per state."""
    X_list, y_list, state_list = [], [], []
    
    for state in df[state_col].unique():
        sdf = df[df[state_col] == state].sort_values('year').reset_index(drop=True)
        if len(sdf) < seq_len + pred_len:
            continue
        
        features = sdf[feature_cols + [target_col]].values
        for i in range(len(sdf) - seq_len - pred_len + 1):
            X_list.append(features[i:i+seq_len, :-1])
            y_list.append(features[i+seq_len, -1])
            state_list.append(state)
    
    return np.array(X_list), np.array(y_list), np.array(state_list)


def walk_forward_split(states, test_years=3):
    """Per-state walk-forward split."""
    train_idx, test_idx = [], []
    for state in np.unique(states):
        idx = np.where(states == state)[0]
        if len(idx) <= test_years:
            continue
        test_idx.extend(idx[-test_years:])
        train_idx.extend(idx[:-test_years])
    return np.array(train_idx), np.array(test_idx)


# ==========================================================
# Models
# ==========================================================
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1)
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                          batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1)
        )
    
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


class BiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.bilstm = nn.LSTM(input_size, hidden_size, num_layers,
                              batch_first=True, dropout=dropout, bidirectional=True)
        self.attention = nn.Linear(hidden_size * 2, 1)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1)
        )
    
    def forward(self, x):
        out, _ = self.bilstm(x)
        attn = torch.softmax(self.attention(out), dim=1)
        context = (out * attn).sum(dim=1)
        return self.fc(context).squeeze(-1)


class TCNModel(nn.Module):
    def __init__(self, input_size, num_channels=[32, 64, 64], 
                 kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        in_ch = input_size
        for out_ch in num_channels:
            layers.extend([
                nn.Conv1d(in_ch, out_ch, kernel_size, padding=kernel_size-1),
                nn.BatchNorm1d(out_ch), nn.ReLU(), nn.Dropout(dropout)
            ])
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(in_ch, 1)
    
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.tcn(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x).squeeze(-1)


# ==========================================================
# Training
# ==========================================================
class MaizeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def train_model(model, train_loader, val_loader, config):
    model = model.to(config.DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), 
                                  lr=config.LEARNING_RATE, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)
    criterion = nn.MSELoss()
    
    best_val, best_state, patience_counter = float('inf'), None, 0
    train_losses, val_losses = [], []
    
    for epoch in range(config.EPOCHS):
        # Train
        model.train()
        train_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(config.DEVICE), yb.to(config.DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        # Val
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(config.DEVICE), yb.to(config.DEVICE)
                val_loss += criterion(model(xb), yb).item()
        val_loss /= len(val_loader)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        scheduler.step(val_loss)
        
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config.PATIENCE:
                break
    
    model.load_state_dict(best_state)
    return model, train_losses, val_losses


def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            preds.append(model(xb.to(device)).cpu().numpy())
    return np.concatenate(preds)


# ==========================================================
# Main Pipeline
# ==========================================================
def main():
    print("=" * 70)
    print("🌽 Maize Yield Forecasting in Nigeria")
    print("=" * 70)
    print(f"🔥 Device: {Config.DEVICE}")
    
    # Load data
    df = pd.read_csv("maize_yearly_analysis.csv")
    df = df.sort_values([Config.STATE_COL, 'year']).reset_index(drop=True)
    print(f"\n📊 Data: {df.shape} | States: {df[Config.STATE_COL].nunique()}")
    
    # Create sequences
    X, y, states = create_sequences(
        df, Config.FEATURE_COLS, Config.TARGET_COL, Config.STATE_COL,
        Config.SEQ_LENGTH, Config.PRED_LENGTH
    )
    print(f"🔧 Sequences: X={X.shape}, y={y.shape}")
    
    # Split
    train_idx, test_idx = walk_forward_split(states, test_years=3)
    print(f"✂️  Train: {len(train_idx)} | Test: {len(test_idx)}")
    
    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test = X[test_idx], y[test_idx]
    
    # Normalize
    scaler_X = StandardScaler()
    n_features = X_train.shape[-1]
    X_train_scaled = scaler_X.fit_transform(
        X_train.reshape(-1, n_features)).reshape(X_train.shape)
    X_test_scaled = scaler_X.transform(
        X_test.reshape(-1, n_features)).reshape(X_test.shape)
    
    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()
    
    # Dataloaders
    train_loader = DataLoader(
        MaizeDataset(X_train_scaled, y_train_scaled),
        batch_size=Config.BATCH_SIZE, shuffle=True
    )
    test_loader = DataLoader(
        MaizeDataset(X_test_scaled, y_test_scaled),
        batch_size=Config.BATCH_SIZE, shuffle=False
    )
    
    # Train models
    input_size = X_train_scaled.shape[-1]
    models_dict = {
        'LSTM': LSTMModel(input_size),
        'GRU': GRUModel(input_size),
        'Bi-LSTM + Attn': BiLSTMAttention(input_size),
        'TCN': TCNModel(input_size),
    }
    
    trained_models, training_histories = {}, {}
    for name, model in models_dict.items():
        print(f"\n🎯 Training {name}...")
        trained, tr_loss, val_loss = train_model(model, train_loader, 
                                                  test_loader, Config)
        trained_models[name] = trained
        training_histories[name] = (tr_loss, val_loss)
        print(f"   ✅ Best val loss: {min(val_loss):.4f}")
    
    # Baselines
    persistence_preds = []
    for state in states[test_idx]:
        state_train_ys = y_train[states[train_idx] == state]
        persistence_preds.append(state_train_ys[-1] if len(state_train_ys) > 0 
                                  else y_train.mean())
    persistence_preds = np.array(persistence_preds)
    
    rf = RandomForestRegressor(n_estimators=200, max_depth=10, 
                                random_state=Config.SEED, n_jobs=-1)
    rf.fit(X_train_scaled.reshape(len(X_train_scaled), -1), y_train_scaled)
    rf_preds = scaler_y.inverse_transform(
        rf.predict(X_test_scaled.reshape(len(X_test_scaled), -1)).reshape(-1, 1)
    ).flatten()
    
    # Evaluate
    def evaluate(y_true, y_pred, name):
        return {
            'name': name,
            'R2': r2_score(y_true, y_pred),
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
        }
    
    results = [
        evaluate(y_test, persistence_preds, "🌱 Persistence Baseline"),
        evaluate(y_test, rf_preds, "🌲 Random Forest"),
    ]
    
    dl_preds = {}
    for name, model in trained_models.items():
        preds = scaler_y.inverse_transform(
            predict(model, test_loader, Config.DEVICE).reshape(-1, 1)
        ).flatten()
        dl_preds[name] = preds
        results.append(evaluate(y_test, preds, f"🧠 {name}"))
    
    ensemble_preds = np.mean(list(dl_preds.values()), axis=0)
    results.append(evaluate(y_test, ensemble_preds, "🎯 Ensemble (DL avg)"))
    
    # Results table
    results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
    print("\n" + "=" * 70)
    print("🏆 Final Results")
    print("=" * 70)
    print(results_df.to_string(index=False))
    
    results_df.to_csv('results/metrics.csv', index=False)
    
    # Summary
    best = results_df.iloc[0]
    persistence_r2 = results_df[
        results_df['name'].str.contains('Persistence')
    ]['R2'].values[0]
    
    print(f"\n🥇 Best model: {best['name']}")
    print(f"📈 R²: {best['R2']:.4f}")
    print(f"📉 Improvement over persistence: {best['R2'] - persistence_r2:+.4f}")
    print("\n✅ Pipeline complete. Results saved to results/")
    print("=" * 70)


if __name__ == "__main__":
    main()

🌽 Maize Yield Forecasting in Nigeria
🔥 Device: cpu

📊 Data: (771, 23) | States: 37
🔧 Sequences: X=(586, 5, 6), y=(586,)
✂️  Train: 475 | Test: 111

🎯 Training LSTM...
   ✅ Best val loss: 0.4915

🎯 Training GRU...
   ✅ Best val loss: 0.5036

🎯 Training Bi-LSTM + Attn...
   ✅ Best val loss: 0.4897

🎯 Training TCN...
   ✅ Best val loss: 0.6027

🏆 Final Results
                  name        R2      MAE     RMSE      MAPE
🌱 Persistence Baseline  0.731938 0.155425 0.232287  7.095847
      🧠 Bi-LSTM + Attn -0.010405 0.355102 0.450977 17.807314
                🧠 LSTM -0.013863 0.356719 0.451748 17.964194
                 🧠 GRU -0.043297 0.360541 0.458259 17.713185
   🎯 Ensemble (DL avg) -0.056405 0.363422 0.461128 17.793298
       🌲 Random Forest -0.171446 0.384195 0.485588 18.069164
                 🧠 TCN -0.255021 0.397497 0.502611 18.502612

🥇 Best model: 🌱 Persistence Baseline
📈 R²: 0.7319
📉 Improvement over persistence: +0.0000

✅ Pipeline complete. Results saved to results/
